# HNSW / small-world graph as sparse attention primitive

This does **not** retrain Sparse Walker. It loads the best ML-1M Walker checkpoint, grows an HNSW graph from the learned concept keys, then asks whether a query derived from the current working-memory state can recover dense-attention targets in only a few sparse graph hops.

The important diagnostics separate **topology** from **navigation**:
- `GLOBAL_HNSW_CONTROL`: can normal HNSW recover exact dense top-10?
- `ORACLE_*`: are targets physically reachable from the current K=8 state within 1–4 hops, ignoring routing?
- `NAVIGATION`: can query-guided beam walking actually find them?
- `DECISION_PANEL`: compact comparison of current graph vs HNSW level-0 vs flattened HNSW with shortcut edges.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, shutil
REPO='/content/Sparsewalker'
BRANCH='agent/walker-swg-attention'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','faiss-cpu'],check=True)
SRC=f'{REPO}/src'
env=os.environ.copy()
env['PYTHONPATH']=SRC + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
env['PYTHONUNBUFFERED']='1'

import torch
print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'bf16',torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
print('Branch',BRANCH)


In [ ]:
cmd=[sys.executable,'-u',f'{REPO}/experiments/run_hnsw_attention_primitive.py',
     '--seed','42',
     '--per-bucket','128',
     '--hnsw-m','8',
     '--ef-construction','160',
     '--max-hops','4',
     '--beams','8','16']
print('RUNNING',' '.join(cmd),flush=True)
subprocess.run(cmd,cwd=REPO,env=env,check=True)


## Compact result
Paste `GLOBAL_HNSW_CONTROL` and `DECISION_PANEL` back into ChatGPT. If the distinction is ambiguous, also paste the HNSW `ORACLE_DENSE` and `NAVIGATION` blocks.

In [ ]:
import json
from pathlib import Path
p=Path('/content/drive/MyDrive/sparsewalker_swg_attention/result.json')
r=json.loads(p.read_text())
print('GLOBAL_HNSW_CONTROL')
print(json.dumps(r['global_hnsw_control'],indent=2))
print('\nDECISION_PANEL')
print(json.dumps(r['decision_panel'],indent=2))
print('\nLONG-HISTORY HNSW FLATTENED, BEAM16')
print(json.dumps(r['graphs']['hnsw_flattened']['navigation_by_history_length']['16']['long_101_200'],indent=2))
